# 01 — Cricket Filtering

This notebook filters the full Reddit dataset down to cricket subreddits only and saves the result back to S3. All later notebooks read from this filtered dataset.

### 1. Setup

Import utilities and start Spark.

In [ ]:
# Add parent folder to path
import sys
sys.path.insert(0, "..")

# Import helpers
from utils import (
    get_spark, load_submissions, load_comments,
    CRICKET_SUBREDDITS,
    S3_CRICKET_SUBMISSIONS, S3_CRICKET_COMMENTS,
    save_to_s3,
)

# Spark functions
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lower, count


### 2. Start Spark

Build the Spark session.

In [ ]:
# Start Spark
spark = get_spark("01_CricketFiltering")
spark


### 3. Show subreddit list

Print the cricket subreddits we will filter to.

In [ ]:
# Print the target list
print(f"Filtering to {len(CRICKET_SUBREDDITS)} cricket subreddits:")
for s in CRICKET_SUBREDDITS:
    print(f"  - {s}")


### 4. Load full submissions

Load the entire submissions dataset from S3.

In [ ]:
# Load submissions
subs = load_submissions(spark)


### 5. Load full comments

Load the entire comments dataset from S3.

In [ ]:
# Load comments
coms = load_comments(spark)


### 6. Filter submissions to cricket subreddits

Keep only rows where the subreddit is in our list (case-insensitive).

In [ ]:
# Lowercase the target list for case-insensitive matching
lower_subs = [s.lower() for s in CRICKET_SUBREDDITS]

# Filter submissions
cricket_subs = subs.filter(lower(col("subreddit")).isin(lower_subs))


### 7. Filter comments to cricket subreddits

Same filter applied to comments.

In [ ]:
# Filter comments
cricket_coms = coms.filter(lower(col("subreddit")).isin(lower_subs))


### 8. Save filtered submissions to S3

Write the filtered submissions back to S3 as parquet, partitioned by yyyy/mm.

In [ ]:
# Save filtered submissions, partitioned by yyyy/mm for fast reads later
save_to_s3(
    cricket_subs,
    S3_CRICKET_SUBMISSIONS,
    mode="overwrite",
    partition_by=["yyyy", "mm"],
)


### 9. Save filtered comments to S3

Write the filtered comments back to S3 as parquet, partitioned by yyyy/mm.

In [ ]:
# Save filtered comments
save_to_s3(
    cricket_coms,
    S3_CRICKET_COMMENTS,
    mode="overwrite",
    partition_by=["yyyy", "mm"],
)


### 10. Verify filtered submissions

Read back the filtered submissions and count rows.

In [ ]:
# Read back and count
verify_subs = spark.read.parquet(S3_CRICKET_SUBMISSIONS)
sub_count = verify_subs.count()
print(f"Filtered submissions saved: {sub_count:,}")


### 11. Verify filtered comments

Read back the filtered comments and count rows.

In [ ]:
# Read back and count
verify_coms = spark.read.parquet(S3_CRICKET_COMMENTS)
com_count = verify_coms.count()
print(f"Filtered comments saved: {com_count:,}")


### 12. Submissions per subreddit

Check post counts in the filtered submissions.

In [ ]:
# Post counts per subreddit
verify_subs.groupBy("subreddit") \
    .agg(count("*").alias("post_count")) \
    .orderBy(col("post_count").desc()) \
    .show(50, truncate=False)


### 13. Comments per subreddit

Check comment counts in the filtered comments.

In [ ]:
# Comment counts per subreddit
verify_coms.groupBy("subreddit") \
    .agg(count("*").alias("comment_count")) \
    .orderBy(col("comment_count").desc()) \
    .show(50, truncate=False)


### 14. Stop Spark

Free up resources.

In [ ]:
# Stop Spark
spark.stop()
